# RQ2: Geometry-guided anchor placement

Locked screening experiment: seed 0 selects two interior anchors from Uniform-100 validation geometry; Geometry-4 seeds 1 and 2 train in parallel under the same 50+50 optimizer-reset protocol. Test evaluation remains sealed until both Geometry-4 checkpoints exist.

In [ ]:
import os, subprocess, sys, time, shutil
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2'
for index in range(torch.cuda.device_count()):
    print(f'GPU {index}: {torch.cuda.get_device_name(index)}')

## Secure checkout
Create a Kaggle secret named `github_token`. `KAGGLE_API_TOKEN` is not used. Enable Internet for repository checkout and CIFAR-100 download.

In [ ]:
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1', 'tabulate>=0.9'], check=True)

## Locate the completed Uniform-100 comparator
Attach the output of `kaggle_s1_extend_to_100.ipynb`. The locator accepts either its direct output tree or its exported ZIP and rejects ambiguous multiple runs.

In [ ]:
import importlib
import rq2_anchor_placement
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)
UNIFORM_INPUT_ROOT = Path('/kaggle/input')
UNIFORM_ROOT = rq2_anchor_placement.find_uniform_100_root(
    UNIFORM_INPUT_ROOT, '/kaggle/working/materialized-uniform-100'
)
print('Validated Uniform-100 root:', UNIFORM_ROOT)

## Create run directory and freeze the seed-0 anchor selection

In [ ]:
from datetime import datetime, timezone
import json, pandas as pd, yaml
from IPython.display import Markdown, display
from torchvision import datasets

config = yaml.safe_load((PROJECT_ROOT / 'configs' / 'kaggle_rq2_anchor.yaml').read_text())
RUN_NAME = datetime.now(timezone.utc).strftime('kaggle-rq2-anchor-%Y%m%d-%H%M%S')
RUN_DIR = Path('/kaggle/working/new-pruning-outputs') / RUN_NAME
config['experiment']['output_dir'] = str(RUN_DIR)
LAUNCH_CONFIG = Path('/kaggle/working/kaggle_rq2_anchor_launch.yaml')
LAUNCH_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
datasets.CIFAR100(root=config['dataset']['root'], train=True, download=True)
datasets.CIFAR100(root=config['dataset']['root'], train=False, download=True)
selection = rq2_anchor_placement.select_geometry_anchors(UNIFORM_ROOT, RUN_DIR / 'protocol')
print('Frozen Geometry-4 anchors:', selection['selected_anchors'])
print('Common holdout:', selection['common_holdout'])
print('Geo/Uniform subnet-FLOPs ratio:', selection['subnet_compute_ratio_geo_to_uniform'])
display(pd.read_csv(RUN_DIR / 'protocol' / 'anchor_training_compute.csv'))
display(pd.read_csv(RUN_DIR / 'protocol' / 'geometry_anchor_selection.csv').head(10))

## Run smoke, two parallel Geometry-4 seeds, then sealed evaluation
The two-epoch smoke result is never used for anchor selection or a scientific decision. Each final seed restarts from scratch, trains epochs 1–50, then performs the matched weights-only SGD/cosine reset for epochs 51–100.

In [ ]:
import scripts.run_rq2_anchor as rq2_runner
rq2_runner = importlib.reload(rq2_runner)
started = time.perf_counter()
result = rq2_runner.run_rq2(LAUNCH_CONFIG, UNIFORM_ROOT, gpu_ids=[0, 1])
print(f'RQ2 completed in {(time.perf_counter() - started) / 3600:.2f} hours')
print(result['decision'])

## Inspect locked outputs

In [ ]:
decision_path = RUN_DIR / 'rq2_decision.json'
display(json.loads(decision_path.read_text()))
if (RUN_DIR / 'rq2_report.md').is_file():
    display(Markdown((RUN_DIR / 'rq2_report.md').read_text()))
    display(pd.read_csv(RUN_DIR / 'rq2_seed_comparison.csv'))
    display(pd.read_csv(RUN_DIR / 'rq2_paired_bootstrap_by_width.csv'))

## Validate and export the complete resumable run

In [ ]:
required = [
    RUN_DIR / 'protocol' / 'selected_anchors.json',
    RUN_DIR / 'protocol' / 'geometry_anchor_selection.csv',
    RUN_DIR / 'protocol' / 'anchor_training_compute.csv',
    RUN_DIR / 'rq2_decision.json',
]
if result['decision'].get('verdict') != 'NO INTERVENTION: geometry selected the uniform anchors':
    required += [
        RUN_DIR / 'rq2_report.md',
        RUN_DIR / 'rq2_seed_comparison.csv',
        RUN_DIR / 'rq2_paired_bootstrap_common_holdout.csv',
        RUN_DIR / 'rq2_paired_bootstrap_by_width.csv',
        RUN_DIR / 'rq2_dense_accuracy_curves.png',
        *[RUN_DIR / 'geo' / f'seed_{seed}' / 'checkpoint.pt' for seed in (1, 2)],
    ]
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
assert not missing, f'Missing required artifacts: {missing}'
files = sorted(path for path in RUN_DIR.rglob('*') if path.is_file())
pd.DataFrame({
    'relative_path': [str(path.relative_to(RUN_DIR)) for path in files],
    'size_bytes': [path.stat().st_size for path in files],
}).to_csv(RUN_DIR / 'artifact_manifest.csv', index=False)
archive = Path(shutil.make_archive(str(Path('/kaggle/working') / RUN_NAME), 'zip', root_dir=RUN_DIR))
print('Validated required artifacts:', len(required))
print('Download:', archive)
archive